# Week 4 — Final Evaluation

1,000-episode simulation harness for evaluating all pricing agents 
(heuristics, Q-Learning, DQN) across full booking seasons.

In [3]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
from pricing_env import PricingEnv
from baseline_agents import FixedPriceAgent, TimeBasedDiscountAgent, DemandBasedAgent


def run_large_scale_evaluation(agent, env, n_episodes=1000, has_reset=False):
    """
    Runs any agent (heuristic, Q-Learning, or DQN) across n_episodes 
    full booking seasons and returns detailed per-episode statistics.
    """
    episode_revenues = []
    episode_sell_through = []

    for ep in range(n_episodes):
        obs, info = env.reset()
        if has_reset:
            agent.reset()
        total_reward = 0
        initial_inventory = env.max_inventory

        done = False
        while not done:
            action = agent.act(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated

        episode_revenues.append(total_reward)
        remaining_inventory = obs[0]
        sell_through = (initial_inventory - remaining_inventory) / initial_inventory
        episode_sell_through.append(sell_through)

    return {
        "revenues": episode_revenues,
        "sell_through_rates": episode_sell_through,
        "mean_revenue": np.mean(episode_revenues),
        "std_revenue": np.std(episode_revenues),
        "mean_sell_through": np.mean(episode_sell_through)
    }

In [4]:
env = PricingEnv()
test_agent = FixedPriceAgent()

results = run_large_scale_evaluation(test_agent, env, n_episodes=1000)
print(f"Mean Revenue: {results['mean_revenue']:.2f}")
print(f"Std Dev: {results['std_revenue']:.2f}")
print(f"Sell-Through Rate: {results['mean_sell_through']*100:.1f}%")

Mean Revenue: 25.75
Std Dev: 3.89
Sell-Through Rate: 46.4%


In [5]:
from baseline_agents import FixedPriceAgent, TimeBasedDiscountAgent, DemandBasedAgent

env = PricingEnv()

agents_to_evaluate = {
    "FixedPrice": (FixedPriceAgent(), False),
    "TimeBasedDiscount": (TimeBasedDiscountAgent(), True),
    "DemandBased": (DemandBasedAgent(), False),
    # "Random": (RandomAgent(), False),        # add if RandomAgent available
    # "QLearning": (q_learning_agent, False),  # add once Member 2's Q-table is loaded
    # "DQN": (dqn_agent, False),               # add once Member 2's DQN checkpoint is loaded
}

all_results = {}
for name, (agent, has_reset) in agents_to_evaluate.items():
    print(f"Running {name}...")
    results = run_large_scale_evaluation(agent, env, n_episodes=1000, has_reset=has_reset)
    all_results[name] = results
    print(f"  Mean Revenue: {results['mean_revenue']:.2f} | Std: {results['std_revenue']:.2f} | Sell-Through: {results['mean_sell_through']*100:.1f}%")

Running FixedPrice...
  Mean Revenue: 25.94 | Std: 3.73 | Sell-Through: 46.7%
Running TimeBasedDiscount...
  Mean Revenue: 18.55 | Std: 1.49 | Sell-Through: 100.0%
Running DemandBased...
  Mean Revenue: 30.33 | Std: 4.16 | Sell-Through: 99.5%


In [6]:
summary_rows = []
for name, res in all_results.items():
    summary_rows.append({
        "Agent": name,
        "Episodes": 1000,
        "Mean Revenue": round(res["mean_revenue"], 2),
        "Std Dev": round(res["std_revenue"], 2),
        "Sell-Through Rate": f"{res['mean_sell_through']*100:.1f}%"
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Agent,Episodes,Mean Revenue,Std Dev,Sell-Through Rate
0,FixedPrice,1000,25.94,3.73,46.7%
1,TimeBasedDiscount,1000,18.55,1.49,100.0%
2,DemandBased,1000,30.33,4.16,99.5%
